In [2]:
import pandas as pd
import import_ipynb
import nbimporter
from utils import prepareTrainingSet3Sec,prepareTrainingSet30Sec,standardization,metrics,Grafico_Before_After,Grafico_Tre_Valori, ConfrontoGrafico_30_e_3, evaluate_Model, prepareTrainingSet30Sec_Arg , prepareTrainingSet3Sec_Arg  #type: ignore
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV  
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import accuracy_score

I file CSV per il training e il test set sono stati creati con successo.


In [3]:
from sklearn.calibration import LabelEncoder
from xgboost import XGBClassifier


X_train, X_test, y_train, y_test= prepareTrainingSet3Sec()

# Converti le etichette delle classi in valori numerici
label_encoder = LabelEncoder()
y_test = label_encoder.fit_transform(y_test)
y_train = label_encoder.fit_transform(y_train)

# utilizzo la funzione di standardizzazzione presente in utils che usa Standard Scaler
X_train,X_test=standardization(X_train,X_test)
# Creare il modello Logistic
model = XGBClassifier(n_estimators=100, learning_rate=0.05,max_depth=6)
model.fit(X_train, y_train)


y_pred = model.predict(X_test)

y_pred_train=model.predict(X_train)
evaluate_Model(y_test,y_pred,y_train,y_pred_train)

---------STATISTICHE TEST----------
Accuratezza: 0.8378
Precision: 0.8391
Recall: 0.8380
F1-score: 0.8372
F2-score: 0.8373
---------STATISTICHE TRAING----------
Accuratezza: 0.9799
Precision: 0.9801
Recall: 0.9798
F1-score: 0.9799
F2-score: 0.9798

=== Analisi ===
Il modello potrebbe soffrire di OVERFITTING (performance molto migliore sul training set).


In [6]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# Assumiamo di avere i tuoi dati X_train, y_train, X_test, y_test
X_train, X_test, y_train, y_test = prepareTrainingSet3Sec()

# Converti le etichette delle classi in valori numerici
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)  # Attenzione: usa fit_transform SOLO sul training set
y_test = label_encoder.transform(y_test)        # Usa transform sul test set

# Utilizza la standardizzazione
X_train, X_test = standardization(X_train, X_test)

# Definiamo la griglia di iperparametri
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.1, 0.01, 0.001],
    'n_estimators': [100, 200, 300],
    'subsample': [0.8, 1],
    'colsample_bytree': [0.8, 1]
}

# Creiamo il modello XGBoost
xgb = XGBClassifier(objective='binary:logistic')

# Modifica opzionale dei tag (se necessario)
# xgb._tags = {**xgb._tags, "classifier": True}  # Solo per versioni recenti di scikit-learn

grid_search = GridSearchCV(xgb, param_grid, scoring='accuracy', cv=5)
grid_search.fit(X_train, y_train)  # Rimuovi la riga che setta estimator_.tags

# Stampa i migliori parametri e il punteggio
print("Best parameters:", grid_search.best_params_)
print("Best cross-validation score:", grid_search.best_score_)

# Valutazione sul test set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

AttributeError: 'super' object has no attribute '__sklearn_tags__'